# Fine-tuning de atendimento: suporte e venda

Este notebook executa a implementação Python do repositório, com os mesmos critérios de validação, treinamento e exportação da CLI. Comece pelo **smoke**, que usa somente dados artificiais e BERT pequeno offline. Para treinar o modelo pré-treinado de verdade, selecione **challenge** e forneça seus JSONL autorizados.

**suporte = 0** e **venda = 1**. Nenhuma chave de API é necessária. Não utilizamos o modelo já ajustado da resolução. Métricas do smoke não são resultados do desafio.

No Colab, selecione GPU em *Ambiente de execução > Alterar tipo de ambiente de execução* para o treinamento completo. O smoke funciona em CPU. O armazenamento do Colab é temporário; exporte o modelo ao terminar.

## 1. Localizar o projeto e instalar

Localmente, instale primeiro o projeto com o extra `notebook` seguindo o README e selecione o interpretador correto. No Colab, a célula abaixo clona o repositório público e instala suas dependências. O projeto fixa a versão de Torch; pacotes de visão/áudio não usados aqui são removidos **somente no Colab** para evitar incompatibilidade. Se houver erro de importação após instalar, reinicie a sessão e execute novamente.

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    PROJECT = Path("/content/fine-tunig")
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "https://github.com/Philipp0Ruan/fine-tuning.git", str(PROJECT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
    if PROJECT is None:
        raise RuntimeError("Abra o notebook dentro do projeto instalado.")
os.chdir(PROJECT)
print("Projeto:", PROJECT)

def run_module(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)], cwd=PROJECT, check=True)


## 2. Escolher o modo

Mantenha `MODE = "smoke"` para verificar o fluxo sem dados oficiais ou download de pesos. Use `"challenge"` somente quando quiser treinar com os JSONL fornecidos e baixar o BERTimbau público. Cada execução usa uma pasta nova para não misturar modelos. O dispositivo automático utiliza CUDA quando disponível.

In [ ]:
from datetime import datetime, timezone
import json
import torch

MODE = "smoke"  # Altere para "challenge" para o treinamento completo.
if MODE not in {"smoke", "challenge"}:
    raise ValueError("MODE deve ser smoke ou challenge")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = PROJECT / "artifacts" / f"notebook-{MODE}-{RUN_ID}"
MODEL_DIR = RUN_DIR / "model" if MODE == "smoke" else RUN_DIR
DATA_DIR = PROJECT / "data" / "raw"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print({"modo": MODE, "dispositivo": DEVICE, "modelo": str(MODEL_DIR)})


## 3. Fornecer os arquivos no modo challenge

Localmente, coloque `treino.jsonl` e `teste.jsonl` em `data/raw/`. No Colab, a célula abre o seletor para enviar os dois arquivos. Cada linha deve ser um objeto com `prompt` e `completion`. Links assinados do Notion podem expirar; o notebook não incorpora essas URLs e não envia os dados para o Hub.

A auditoria seguinte exige ambas as classes, rejeita conflitos e interseção treino/teste, remove duplicatas internas idênticas e reserva validação somente do treino.

In [ ]:
if MODE == "challenge":
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        required = {"treino.jsonl", "teste.jsonl"}
        if not required.issubset(uploaded):
            raise ValueError("Selecione os dois arquivos: treino.jsonl e teste.jsonl")
        for name in required:
            (DATA_DIR / name).write_bytes(uploaded[name])
        del uploaded
    for name in ("treino.jsonl", "teste.jsonl"):
        if not (DATA_DIR / name).is_file():
            raise FileNotFoundError(DATA_DIR / name)
    run_module("retail_intent.cli", "validate", "--train-file", DATA_DIR / "treino.jsonl",
               "--test-file", DATA_DIR / "teste.jsonl")
else:
    print("Modo smoke: os dados artificiais serão gerados na próxima etapa.")


## 4. Treinar e exportar

O modo challenge ajusta o BERTimbau por cinco épocas, escolhe o checkpoint pelo F1 macro de validação e calcula o teste somente no fim. Os padrões são batch 8, taxa `2e-5`, 128 tokens e semente 42. Para repetir a origem do modelo exatamente, substitua `MODEL_REVISION = "main"` por um hash de commit do Hub. Reduza batch em caso de memória insuficiente.

O smoke cria um BERT pequeno com pesos aleatórios e executa duas épocas, verificando a alteração dos parâmetros. Esse ensaio não mede qualidade de português nem prontidão comercial.

In [ ]:
if MODE == "smoke":
    import re
    from transformers import BertConfig, BertForSequenceClassification, BertTokenizerFast, set_seed
    from retail_intent.config import ID2LABEL, LABEL2ID

    set_seed(42)
    torch.set_num_threads(2)
    demo_data = RUN_DIR / "data"
    demo_data.mkdir(parents=True)
    products = ["cafeteira", "geladeira", "forno", "aspirador", "lavadora", "televisor"]
    templates = {
        "venda": ["Quero comprar {p}", "Gostaria de adquirir {p}", "Tem {p} para vender?", "Como faço para encomendar {p}?"],
        "suporte": ["Meu produto {p} parou de funcionar", "Como configurar {p}?", "Preciso de ajuda com defeito em {p}", "Onde encontro o manual de {p}?"],
    }
    train_rows = [{"prompt": template.format(p=product), "completion": label}
                  for label, variants in templates.items() for template in variants for product in products]
    test_rows = [
        {"prompt": "Desejo finalizar a compra de uma torradeira", "completion": "venda"},
        {"prompt": "Vocês vendem ventilador para entrega?", "completion": "venda"},
        {"prompt": "Estou interessado em adquirir um liquidificador", "completion": "venda"},
        {"prompt": "Quero encomendar um micro-ondas novo", "completion": "venda"},
        {"prompt": "Minha torradeira não está ligando", "completion": "suporte"},
        {"prompt": "Como limpar o filtro do ventilador?", "completion": "suporte"},
        {"prompt": "O liquidificador faz barulho estranho", "completion": "suporte"},
        {"prompt": "Como ajustar a potência do micro-ondas?", "completion": "suporte"},
    ]
    for name, rows in (("treino.jsonl", train_rows), ("teste.jsonl", test_rows)):
        (demo_data / name).write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
    base = RUN_DIR / "tiny-base"
    base.mkdir()
    vocabulary = sorted({token for row in train_rows for token in re.findall(r"\w+|[^\w\s]", row["prompt"].lower())})
    tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + vocabulary
    (base / "vocab.txt").write_text("\n".join(tokens) + "\n", encoding="utf-8")
    tokenizer = BertTokenizerFast(vocab_file=str(base / "vocab.txt"), do_lower_case=True)
    tokenizer.save_pretrained(base)
    initial_model = BertForSequenceClassification(BertConfig(
        vocab_size=len(tokens), hidden_size=32, num_hidden_layers=1, num_attention_heads=2,
        intermediate_size=64, max_position_embeddings=64, num_labels=2,
        id2label=ID2LABEL, label2id=LABEL2ID))
    initial_weights = initial_model.classifier.weight.detach().clone()
    initial_model.save_pretrained(base, safe_serialization=True)
    print("Base artificial e dados criados somente para demonstração.")


In [ ]:
MODEL_REVISION = "main"
if MODE == "smoke":
    from retail_intent.train import TrainConfig, run_training
    demo_metrics = run_training(TrainConfig(
        train_file=demo_data / "treino.jsonl", test_file=demo_data / "teste.jsonl",
        output_dir=MODEL_DIR, model_name=str(base.resolve()), epochs=2, batch_size=8,
        learning_rate=1e-3, max_length=32, device="cpu", purpose="smoke"))
else:
    run_module("retail_intent.cli", "train",
               "--train-file", DATA_DIR / "treino.jsonl", "--test-file", DATA_DIR / "teste.jsonl",
               "--output-dir", MODEL_DIR, "--model-name", "neuralmind/bert-base-portuguese-cased",
               "--revision", MODEL_REVISION, "--epochs", 5, "--batch-size", 8,
               "--learning-rate", 2e-5, "--max-length", 128, "--validation-size", 0.2,
               "--seed", 42, "--device", DEVICE)


## 5. Ler resultados

Compare F1 macro e métricas por classe com a baseline. A matriz usa linhas reais e colunas previstas na ordem `suporte`, `venda`. Não ajuste o treinamento olhando o teste. No smoke, qualquer número abaixo é apenas diagnóstico de execução.

In [ ]:
from IPython.display import display

metrics = json.loads((MODEL_DIR / "metrics.json").read_text(encoding="utf-8"))
manifest = json.loads((MODEL_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Propósito:", manifest["purpose"])
print("Melhor F1 macro na validação:", metrics["best_validation_f1_macro"])
display({"transformer_test": metrics["test"], "tfidf_test": metrics["baseline_tfidf"]["test"],
         "majority_test": metrics["baseline_majority_test"]})


## 6. Prever novas mensagens

As probabilidades softmax não são confiança calibrada. O sinal `needs_review` usa limiar 0,75 como ponto inicial, a ser validado com dados representativos. O modelo smoke é permitido aqui explicitamente para demonstração.

In [ ]:
from retail_intent.predict import IntentClassifier

classifier = IntentClassifier(MODEL_DIR, device=DEVICE, threshold=0.75, allow_smoke=MODE == "smoke")
if MODE == "smoke":
    print("Pesos alterados:", not torch.equal(initial_weights, classifier.model.classifier.weight.detach().cpu()))
display(classifier.predict(["Gostaria de comprar uma cafeteira", "Minha cafeteira não aquece a água"]))


## 7. Exportar antes de encerrar a sessão

O ZIP abaixo contém pesos, tokenizer, configuração, manifesto e relatórios, sem dados de treino, checkpoints, `.env` ou testes. Guarde-o em local autorizado. Para servir, extraia em `artifacts/` e aponte `MODEL_DIR` para a pasta que contém `manifest.json`. O artefato smoke permanece bloqueado por padrão pela API.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

archive = PROJECT / "artifacts" / f"modelo-{MODE}-{RUN_ID}.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as output:
    for file in MODEL_DIR.iterdir():
        if file.is_file() and file.name != "training_args.bin":
            output.write(file, arcname=file.name)
print("Arquivo:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))


## Próximos passos

Siga o README para iniciar `python -m retail_intent.api` e `npm --prefix gateway start`, ou use o Compose depois de treinar. Para atendimento real, revise erros, calibre o limiar e avalie dados representativos. Os testes sintéticos não substituem essa avaliação.

Referências: [BERTimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased), [classificação com Transformers 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/en/tasks/sequence_classification), [repositório](https://github.com/Philipp0Ruan/fine-tuning).